# 01 — Constraint Analysis

Define the design point before any hardware is fixed. Outputs feed the
mass and propulsion notebooks.

**Inputs:** `mission/mission_params.yaml`, `airframe/airframe_params.yaml`

**Outputs:** required T/W, disc loading vs endurance trade curve, minimum
total thrust at target AUW.


In [ ]:
"""Add the repo root to sys.path so `import models` works from any subdir."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

FIG_DIR = REPO_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
with open(REPO_ROOT / "mission" / "mission_params.yaml") as f:
    mission = yaml.safe_load(f)
with open(REPO_ROOT / "airframe" / "airframe_params.yaml") as f:
    airframe = yaml.safe_load(f)

mission, airframe


## Required thrust at the design point


In [ ]:
from models import propulsion as prop

auw_kg = mission["design_point"]["target_auw_kg"]
tw = mission["design_point"]["thrust_to_weight_ratio"]
n_motors = airframe["airframe"]["num_motors"]
prop_d_in = airframe["airframe"]["max_prop_diameter_in"]
alt_m = mission["mission"]["altitude_asl_m"]

rho = prop.isa_density(alt_m, mission["environment"]["temperature_offset_c"])
T_per_motor_n = prop.thrust_per_motor(auw_kg, n_motors, tw)
T_total_n = T_per_motor_n * n_motors
A_disc = prop.disc_area_m2(prop_d_in, n_motors)
DL = prop.disc_loading(auw_kg * prop.G, A_disc)

print(f"Target AUW:                 {auw_kg:.2f} kg")
print(f"Air density @{alt_m} m:      {rho:.4f} kg/m^3")
print(f"Required T/W:               {tw:.2f}")
print(f"Total disc area:            {A_disc:.3f} m^2")
print(f"Hover disc loading:         {DL:.1f} N/m^2")
print(f"Required peak thrust/motor: {T_per_motor_n:.2f} N "
      f"({T_per_motor_n/prop.G*1000:.0f} gf)")
print(f"Required peak thrust total: {T_total_n:.2f} N")


## Disc loading vs hover power

Sweeps prop diameter to show the efficiency benefit of larger props at a
fixed AUW. Lower disc loading → less induced power → longer endurance.


In [ ]:
diameters_in = np.linspace(13, 22, 40)
A = prop.disc_area_m2(diameters_in, n_motors)
T_total = auw_kg * prop.G  # hover thrust
P_hover_w = np.array([
    prop.prop_power_estimate(T_total / n_motors, a / n_motors, rho) * n_motors
    for a in A
])
DL_arr = T_total / A

fig, ax1 = plt.subplots(figsize=(7, 4.5))
ax1.plot(diameters_in, P_hover_w, color="C0", label="Hover shaft power")
ax1.set_xlabel("Prop diameter [in]")
ax1.set_ylabel("Total hover shaft power [W]", color="C0")
ax1.tick_params(axis="y", labelcolor="C0")
ax1.axvline(prop_d_in, ls="--", color="grey", alpha=0.6,
            label=f"X6 max prop ({prop_d_in}\")")

ax2 = ax1.twinx()
ax2.plot(diameters_in, DL_arr, color="C3", label="Disc loading")
ax2.set_ylabel("Hover disc loading [N/m^2]", color="C3")
ax2.tick_params(axis="y", labelcolor="C3")

ax1.set_title("Hover power vs prop diameter @ AUW = "
              f"{auw_kg} kg, {n_motors} motors, {alt_m} m ASL")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_disc_loading_vs_power.png", dpi=140)
plt.show()


## Optional: ADRpy constraint diagram

If `ADRpy` is installed it can be used here to plot multirotor T/W vs
disc loading constraints (climb, ceiling, gust margin). The block below
is import-guarded so the notebook still runs end-to-end without it.


In [ ]:
try:
    import ADRpy  # noqa: F401
    print("ADRpy available — extend this cell with constraint curves.")
except ModuleNotFoundError:
    print("ADRpy not installed; using simplified constraint plot above.")


## Outputs handed to notebook 02


In [ ]:
constraints_summary = {
    "auw_kg": auw_kg,
    "thrust_to_weight": tw,
    "thrust_total_n": T_total_n,
    "thrust_per_motor_n": T_per_motor_n,
    "disc_loading_n_m2": float(DL),
    "air_density_kg_m3": float(rho),
}
constraints_summary
